# Treasuries data workflow

`pull_treasuries` returns CRSP daily treasuries for a date range. Each business day is cached to `data/raw/treasuries/crsp_treasuries_YYYYMMDD.parquet`, so WRDS is only hit for days not yet cached.

In [1]:
! uv sync

Resolved 37 packages in 13ms
Uninstalled 11 packages in 87ms
 - appnope==1.0.0
 - comm==0.2.3
 - debugpy==1.8.22
 - ipykernel==7.4.0
 - jupyter-client==8.10.0
 - jupyter-core==5.9.1
 - nest-asyncio2==1.7.4
 - packaging==26.3
 - platformdirs==4.14.3
 - pyzmq==27.2.0
 - tornado==6.6


In [2]:
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `src` imports

from src.data import pull_treasuries

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Pull

First call queries WRDS and caches per day. Later calls read the cache. `force_refresh=True` re-pulls.

In [3]:
df = pull_treasuries("2024-01-01", "2024-01-31")
df.head()

2024-01-02: 440 rows loaded
2024-01-03: 440 rows loaded
2024-01-04: 439 rows loaded
2024-01-05: 439 rows loaded
2024-01-08: 439 rows loaded
2024-01-09: 439 rows loaded
2024-01-10: 440 rows loaded
2024-01-11: 441 rows loaded
2024-01-12: 441 rows loaded
2024-01-16: 439 rows loaded
2024-01-17: 439 rows loaded
2024-01-18: 440 rows loaded
2024-01-19: 440 rows loaded
2024-01-22: 440 rows loaded
2024-01-23: 440 rows loaded
2024-01-24: 441 rows loaded
2024-01-25: 442 rows loaded
2024-01-26: 443 rows loaded
2024-01-29: 443 rows loaded
2024-01-30: 443 rows loaded
2024-01-31: 440 rows loaded


,cusip,date,security_type,coupon_rate,dated_date,maturity_date,bid_price,ask_price,mid_price,mid_price_flag,accrued_interest,daily_yield,annual_yield,duration_days,duration_years,daily_return,total_outstanding,public_outstanding
0,912810ES,2024-01-02,1,7.5,1994-05-15,2024-11-15,102.5625,102.59375,102.578125,M,0.989011,0.00012,0.04367,311.443616,0.85327,0.001099,9604.0,3478.0
1,912810ET,2024-01-02,1,7.625,1995-02-15,2025-02-15,103.46875,103.5,103.484375,M,2.900815,0.000119,0.043328,390.532662,1.069952,0.000192,9509.0,3333.0
2,912810EV,2024-01-02,1,6.875,1995-08-15,2025-08-15,103.703125,103.734375,103.71875,M,2.615489,0.000121,0.044164,556.361616,1.524278,-0.000179,11187.0,3356.0
3,912810EW,2024-01-02,1,6.0,1996-02-15,2026-02-15,103.289062,103.320312,103.304688,M,2.282609,0.000118,0.042941,724.389519,1.984629,-0.001083,12838.0,6194.0
4,912810EX,2024-01-02,1,6.75,1996-08-15,2026-08-15,106.359375,106.390625,106.375,M,2.567935,0.000113,0.041115,873.970564,2.39444,-0.002047,8810.0,3670.0


## Sanity checks

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9248 entries, 0 to 9247
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   cusip               9248 non-null   string        
 1   date                9248 non-null   datetime64[ns]
 2   security_type       9248 non-null   int64         
 3   coupon_rate         9248 non-null   Float64       
 4   dated_date          9248 non-null   datetime64[ns]
 5   maturity_date       9248 non-null   datetime64[ns]
 6   bid_price           9248 non-null   Float64       
 7   ask_price           9248 non-null   Float64       
 8   mid_price           9248 non-null   Float64       
 9   mid_price_flag      9248 non-null   string        
 10  accrued_interest    9248 non-null   Float64       
 11  daily_yield         8154 non-null   Float64       
 12  annual_yield        8154 non-null   Float64       
 13  duration_days       8154 non-null   Float64     

In [5]:
df.groupby("date").size().describe()

count     21.000000
mean     440.380952
std        1.359272
min      439.000000
25%      439.000000
50%      440.000000
75%      441.000000
max      443.000000
dtype: float64

In [6]:
df.groupby("security_type").size()

security_type
1     1995
2     5076
4     1083
11     462
12     632
dtype: int64

In [7]:
df[["bid_price", "ask_price", "mid_price", "annual_yield", "duration_years"]].describe()

,bid_price,ask_price,mid_price,annual_yield,duration_years
count,9248.0,9248.0,9248.0,8154.0,8154.0
mean,93.789519,93.830988,93.810254,0.044272,4.960763
std,9.757897,9.74448,9.75117,0.004781,5.464816
min,49.984375,50.078125,50.03125,0.001239,0.00274
25%,91.394531,91.429688,91.410156,0.040437,0.909847
50%,97.171875,97.195312,97.182151,0.042941,2.761797
75%,99.332031,99.359375,99.351562,0.047541,5.960571
max,113.101562,113.148438,113.125,0.077129,20.654885


In [ ]:
df.maturity_date - df.dated_date

,cusip,date,security_type,coupon_rate,dated_date,maturity_date,bid_price,ask_price,mid_price,mid_price_flag,accrued_interest,daily_yield,annual_yield,duration_days,duration_years,daily_return,total_outstanding,public_outstanding
0,912810ES,2024-01-02,1,7.5,1994-05-15,2024-11-15,102.5625,102.59375,102.578125,M,0.989011,0.00012,0.04367,311.443616,0.85327,0.001099,9604.0,3478.0
1,912810ET,2024-01-02,1,7.625,1995-02-15,2025-02-15,103.46875,103.5,103.484375,M,2.900815,0.000119,0.043328,390.532662,1.069952,0.000192,9509.0,3333.0
2,912810EV,2024-01-02,1,6.875,1995-08-15,2025-08-15,103.703125,103.734375,103.71875,M,2.615489,0.000121,0.044164,556.361616,1.524278,-0.000179,11187.0,3356.0
3,912810EW,2024-01-02,1,6.0,1996-02-15,2026-02-15,103.289062,103.320312,103.304688,M,2.282609,0.000118,0.042941,724.389519,1.984629,-0.001083,12838.0,6194.0
4,912810EX,2024-01-02,1,6.75,1996-08-15,2026-08-15,106.359375,106.390625,106.375,M,2.567935,0.000113,0.041115,873.970564,2.39444,-0.002047,8810.0,3670.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9243,91282CJV,2024-01-31,2,4.25,2024-01-31,2026-01-31,100.0625,100.097656,100.080078,M,0.0,0.000114,0.041587,708.477993,1.941036,0.002426,59998.0,<NA>
9244,91282CJT,2024-01-31,2,4.0,2024-01-15,2027-01-15,100.023438,100.058594,100.041016,M,0.175824,0.000108,0.039417,1027.633652,2.815435,0.003827,51998.0,<NA>
9245,91282CJW,2024-01-31,2,4.0,2024-01-31,2029-01-31,100.714844,100.761719,100.738281,M,0.0,0.000104,0.037961,1674.503505,4.587681,0.006007,61000.0,<NA>
9246,91282CJX,2024-01-31,2,4.0,2024-01-31,2031-01-31,100.71875,100.765625,100.742188,M,0.0,0.000105,0.03838,2256.597804,6.18246,0.007816,41000.0,<NA>


In [8]:
df.cusip.unique()

<StringArray>
['912810ES', '912810ET', '912810EV', '912810EW', '912810EX', '912810EY',
 '912810EZ', '912810FA', '912810FB', '912810FE',
 ...
 '91282CJY', '912797JX', '912797JT', '912797JR', '912797JY', '91282CJV',
 '912797JU', '91282CJW', '91282CJX', '912797JZ']
Length: 453, dtype: string